## Book Translation Pipeline

**Goal**  
Convert scanned PDF pages (Hindi/Sanskrit) into a cleaned, chapter-aware English manuscript ready for final formatting.

**Inputs**  
- Original PDF file (scanned pages).  
- OCR output saved as `output_hindi.txt` and intermediate lists like `all_text`.

**Outputs**  
- `clean_pages` list with one cleaned paragraph per page.  
- `final_text` ready for chapter splitting, cleaning, and export.

**High level steps**  
1. Load libraries and tools for OCR and PDF conversion.  
2. Extract text from PDF pages using `pytesseract` and `pdf2image`.  
3. Clean and normalize OCR output into `clean_pages`.  
4. Detect chapters, apply title overrides, and assemble `final_text`.  
5. Insert chapter breaks and export for final PDF formatting.


# Import Translator library

This cell imports translation utilities. Keep this at the top so translation functions are available later if needed.


In [1]:
# Import translator (optional)
# The Translator can be used later for quick translation checks.
from googletrans import Translator


# Load core OCR and image libraries

This cell imports the main libraries used for PDF→image conversion and OCR:
- `pdf2image` to convert PDF pages to images.
- `pytesseract` to run OCR on images.
- `PIL.Image` for image handling.


In [10]:
# Load OCR and image libraries
import pytesseract
import pdf2image
from PIL import Image

# Quick confirmation message to ensure imports succeeded
print("Libraries loaded successfully!")


Libraries loaded successfully!


# Optional helper functions

Use this cell to add small helper functions (e.g., safe file readers, logging helpers) if needed.


In [14]:
# Demo OCR on a single page

#This cell demonstrates converting the first page of the PDF to an image and running OCR on it.
#It is useful for verifying language settings and OCR quality before processing the whole document.


# Convert and OCR the first page to verify OCR settings
from pdf2image import convert_from_path


# Path to your PDF (update to your local path)
pdf_path = r"C:\Users\hisha\Desktop\Me\Work life\Projects\OCR_test\Vaidika-Vishva-darshan.pdf"

# Convert ONLY the first page to check OCR output
pages = convert_from_path(
    pdf_path,
    dpi=300,
    first_page=1,
    last_page=1,
    poppler_path=r"C:\Users\hisha\Desktop\Me\Work life\Projects\OCR_test\poppler-25.12.0\Library\bin"
)

# OCR the first page using Hindi language pack
first_page = pages[0]
text = pytesseract.image_to_string(first_page, lang="hin")

print("Extracted text from page 1:")
print(text)


Extracted text from page 1:
हिन्दू विश्वविद्यालय श्रीमती सरदार कुँवर बाई फण्ड ग्रन्थमाला क्‍
प्गाहा परशाएगणंत]शैगु8 जयशी 90१0॥
त॒ग्राप्तक्ष' ठग या शाओेएं 0श7४8

वि
एप, ॥

बैदिक विश्वदर्श

लेखक
पण्डित दरिशंकर जोशी, एम्‌० एु०

प्रधान संपादक
. श्री वासुदेव भरण अग्रवाल

वैक्रमाब्द २०२२ ] गे [ प्रथम संस्करण



In [19]:
# Detect total number of pages

# pdf2image does not directly return the page count. 
# This cell uses `pdfinfo` (from poppler) to query the PDF metadata and extract the total page count.

# Determine total pages using pdfinfo (poppler)
from tqdm import tqdm

pdf_path = r"C:\Users\hisha\Desktop\Me\Work life\Projects\OCR_test\Vaidika-Vishva-darshan.pdf"
poppler_path = r"C:\Users\hisha\Desktop\Me\Work life\Projects\OCR_test\poppler-25.12.0\Library\bin"

# Use pdfinfo to get page count
import subprocess, re

cmd = f'"{poppler_path}\\pdfinfo.exe" "{pdf_path}"'
out = subprocess.check_output(cmd, shell=True).decode("utf-8")
match = re.search(r"Pages:\s+(\d+)", out)
total_pages = int(match.group(1))
print(f"Total pages detected: {total_pages}")


Total pages detected: 980


# Extract text from every page

This cell converts each PDF page to an image and runs OCR on it. The results are appended to `all_text` and saved to `output_hindi.txt`.
Note: This step can be slow for large PDFs; use `tqdm` for progress feedback.


In [20]:
# Extract OCR text from every page and save to a file
all_text = []

for page_num in tqdm(range(1, total_pages + 1)):
    pages = convert_from_path(
        pdf_path,
        dpi=300,
        first_page=page_num,
        last_page=page_num,
        poppler_path=poppler_path
    )
    text = pytesseract.image_to_string(pages[0], lang="hin")
    all_text.append(text)

# Save OCR output to a UTF-8 text file for later processing
with open("output_hindi.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_text))

print("Extraction complete!")



100%|██████████| 980/980 [1:39:33<00:00,  6.10s/it]  

Extraction complete!


In [23]:
# Confirm output file path
# Print the absolute path to the saved OCR output so you can open it in an editor.

print(os.path.abspath("output_hindi.txt"))

C:\Users\hisha\Desktop\Me\Work life\Projects\Language\output_hindi.txt


# Clean OCR pages into single paragraphs

This cell:
- Splits each page into lines
- Strips whitespace and removes empty lines and page numbers
- Merges lines into a single paragraph per page
- Normalizes whitespace
Result is stored in `clean_pages`.


In [28]:
# Clean OCR output into one paragraph per page
import re

clean_pages = []

for page in tqdm(all_text, desc="Cleaning pages"):
    # Split into individual lines and strip whitespace
    lines = page.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Skip pure page numbers like "23"
        if line.isdigit():
            continue

        cleaned_lines.append(line)

    # Merge lines into one paragraph and normalize spaces
    merged = " ".join(cleaned_lines)
    merged = re.sub(r"\s+", " ", merged).strip()

    clean_pages.append(merged)

print("Cleaning complete!")


Cleaning pages: 100%|██████████| 980/980 [00:00<00:00, 1842.68it/s]

Cleaning complete!


In [33]:
# Uncomment and Print a sample cleaned page to verify results
# print(clean_pages[0])

In [30]:
print(clean_pages[500])

अध्याय ५६ अहोरात्रवाद वेदिकों का अहोरात्रवाद, द्शन का बृहत्‌ प्रकाश स्तम्भ है। यह एक प्रकार का नहीं वरन्‌ अनेक प्रकार का है) पर यह अनेक प्रकार का केवछ नाम और व्याख्या शेली के अनुसार है, तत्त्वतः यह केवल एक ही प्रकार १--वबेदिक अहोरात्र- का है। क्योंकि इसके नाना प्रकार, वेद्कि दशेन के पूरे पचास बाद के भेद तत्त्वों के विभागों का ही वर्णन करते हैं। वेदिक दर्शन को ओर उसके विभाजनों को विभिन्‍न संज्ञाओं के अहोरात्रों के नाम से पुकार कर उनकी व्याख्या उन नामों के अनुसार मात्र की गई है । जेसे यह अहो- राक्षबाद चार प्रकार का मुख्यतः माना जाता है (१) मानुष अहोराज्रवाद (२) पितृ अहोरात्रवाद (३) देवत अहोरात्रवाद (४) त्राह्म अहोरातन्रवाद । इस प्राह्म अहोरात्र बाद को ७२, ७२ युगों के विभाजन से मन्वन्तरों में भी विभाजित किया जाता है । ये मन्वन्तर सप्तकों के महर्षियों या मुख्य ब्रह्मों के प्रतिनिधि होते हैं । चारों अहो- रात्रवाद संबत्सर त्रह्म कहलाते हैं । मानुष अहोरात्रवाद पूरे पचास तत्त्वों को दो भाग दिन और रात में विभाजित करता है। इसका मध्य बिन्दु २५ वां तत्त्व नूषदू या मनुष्य सप्तक में आता है, अतः यह म

In [32]:
#print(clean_pages[250])

### Paragraph Preserving Cleaning

**Purpose**  
Group OCR lines into natural paragraphs while preserving sentence boundaries. This step:
- removes empty lines and page numbers,
- merges lines into sentences,
- splits paragraphs when a sentence-ending punctuation mark is found.

**Inputs**  
- `all_text`: list of raw OCR page strings.

**Outputs**  
- `clean_paragraph_pages`: list of page strings where paragraphs are separated by a blank line.


In [34]:
# Build paragraph-preserving pages from raw OCR output
from tqdm import tqdm
import re

clean_paragraph_pages = []

for page in tqdm(all_text, desc="Cleaning with paragraphs"):
    # Split page into lines and trim whitespace
    lines = page.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Skip pure page numbers like "23"
        if line.isdigit():
            continue

        cleaned_lines.append(line)

    # Build paragraphs by buffering lines until a sentence-ending punctuation appears
    paragraphs = []
    buffer = ""

    for line in cleaned_lines:
        # Append line to buffer (with a space if buffer already has content)
        buffer = f"{buffer} {line}" if buffer else line

        # If the line ends with a sentence terminator, finalize the paragraph
        if line.endswith("।") or line.endswith("."):
            paragraphs.append(buffer.strip())
            buffer = ""

    # Add any leftover buffer as a paragraph
    if buffer:
        paragraphs.append(buffer.strip())

    # Join paragraphs with double newlines to preserve paragraph breaks
    page_text = "\n\n".join(paragraphs)
    clean_paragraph_pages.append(page_text)

print("Paragraph-preserving cleaning complete!")


Cleaning with paragraphs: 100%|██████████| 980/980 [00:00<00:00, 6666.00it/s]

Paragraph-preserving cleaning complete!


In [35]:
# Print a sample cleaned page to verify paragraph preservation
# Adjust the index as needed to inspect other pages
print(clean_paragraph_pages[500])


अध्याय ५६ अहोरात्रवाद वेदिकों का अहोरात्रवाद, द्शन का बृहत्‌ प्रकाश स्तम्भ है। यह एक प्रकार का नहीं वरन्‌ अनेक प्रकार का है) पर यह अनेक प्रकार का केवछ नाम और व्याख्या शेली के अनुसार है, तत्त्वतः यह केवल एक ही प्रकार १--वबेदिक अहोरात्र- का है। क्योंकि इसके नाना प्रकार, वेद्कि दशेन के पूरे पचास बाद के भेद तत्त्वों के विभागों का ही वर्णन करते हैं। वेदिक दर्शन को ओर उसके विभाजनों को विभिन्‍न संज्ञाओं के अहोरात्रों के नाम से पुकार कर उनकी व्याख्या उन नामों के अनुसार मात्र की गई है । जेसे यह अहो- राक्षबाद चार प्रकार का मुख्यतः माना जाता है (१) मानुष अहोराज्रवाद (२) पितृ अहोरात्रवाद (३) देवत अहोरात्रवाद (४) त्राह्म अहोरातन्रवाद । इस प्राह्म अहोरात्र बाद को ७२, ७२ युगों के विभाजन से मन्वन्तरों में भी विभाजित किया जाता है ।

ये मन्वन्तर सप्तकों के महर्षियों या मुख्य ब्रह्मों के प्रतिनिधि होते हैं । चारों अहो- रात्रवाद संबत्सर त्रह्म कहलाते हैं ।

मानुष अहोरात्रवाद पूरे पचास तत्त्वों को दो भाग दिन और रात में विभाजित करता है। इसका मध्य बिन्दु २५ वां तत्त्व नूषदू या मनुष्य सप्तक में आता है, अतः यह

In [ ]:
### Optional: Additional Paragraph Checks

# For extra checks or small utilities related to paragraph processing (e.g., counting paragraphs per page).


In [37]:
# Example: test googletrans on a sample page (commented out)
# from googletrans import Translator
# translator = Translator()

# sample = clean_paragraph_pages[500]  # or any page you want
# translated = translator.translate(sample, dest="en").text

# print(translated)



### Load Environment Variables

**Purpose**  
Load Azure translator credentials from a `.env` file. Keep secrets out of the notebook and use environment variables for secure access.


In [44]:
# Load environment variables for Azure Translator
import os
from dotenv import load_dotenv

load_dotenv()

AZURE_TRANSLATOR_KEY = os.getenv("AZURE_TRANSLATOR_KEY")
AZURE_TRANSLATOR_ENDPOINT = os.getenv("AZURE_TRANSLATOR_ENDPOINT")
AZURE_TRANSLATOR_REGION = os.getenv("AZURE_TRANSLATOR_REGION")

# Quick check (uncomment to debug)
# print(AZURE_TRANSLATOR_KEY is not None, AZURE_TRANSLATOR_ENDPOINT is not None)


### Initialize Azure Translation Client

**Purpose**  
Create an authenticated `TextTranslationClient` instance using the loaded environment variables. This client will be used to translate pages from Hindi to English.


In [50]:
# Initialize Azure Text Translation client
from azure.ai.translation.text import TextTranslationClient
from azure.core.credentials import AzureKeyCredential

client = TextTranslationClient(
    endpoint=AZURE_TRANSLATOR_ENDPOINT,
    credential=AzureKeyCredential(AZURE_TRANSLATOR_KEY),
    region=AZURE_TRANSLATOR_REGION
)


In [55]:
# Translate a single page using Azure Text Translation
def translate_page(text: str) -> str:
    
    #Translate `text` from Hindi to English using Azure Text Translation.
    #Returns the translated English string.
    
    response = client.translate(
        body=[{"text": text}],
        to_language=["en"],
        from_language="hi"
    )
    return response[0].translations[0].text


In [57]:
# Example test for translate_page (commented out)

# sample = clean_paragraph_pages[500]  # or any page you want
# translated = translate_page(sample)
# print(translated)


In [59]:
# Import time for pacing API requests or adding delays between calls
import time

### Original translation loop (initial experiment)

**What this cell did**  
- Iterated over the first 300 pages and called `translate_page` for each.  
- Appended each translated page to `translated_pages`.

**Why it failed**  
- The loop sent requests too quickly and hit the translator service rate limits (HTTP 429).  
- The error trace is preserved below for debugging.

**How to use this cell now**  
- Keep it for history. Do not run it again without adding rate limiting, retries, or batching.
c.


In [58]:
# Original naive translation loop (kept for historical record)
# WARNING: This naive loop can trigger API rate limits (HTTP 429) if run as-is.

translated_pages = []

for i in tqdm(range(300), desc="Translating pages"):
    hindi_text = clean_paragraph_pages[i]
    english_text = translate_page(hindi_text)
    translated_pages.append(english_text)


Translating pages:   7%|▋         | 20/300 [00:13<03:09,  1.48it/s]


HttpResponseError: (429001) The server rejected the request because the client has exceeded request limits.
Code: 429001
Message: The server rejected the request because the client has exceeded request limits.

In [ ]:
# Print the last translated page index (useful when translating in batches)
# Assumes `translated_pages` is a list of translated page strings
print("Last translated page index:", len(translated_pages) - 1)

### Incremental resume attempt with basic retry

**What this cell attempted**  
- Resumed translation from the last saved index and added a simple `try/except` with a short sleep on error.  
- Introduced a small `time.sleep(0.5)` to slow requests.

**Observed behavior**  
- Still hit 429 errors. The simple `break` on exception stops the batch early, which is useful to avoid repeated failures but requires manual restart.

**Recommendation**  
- Keep this cell as an experiment. Use the robust resumable loop later for production runs.


In [62]:
# Incremental attempt with minimal backoff (kept for record)
start_page = len(translated_pages)   # should be 20
end_page = 300

for i in tqdm(range(start_page, end_page), desc="Translating pages"):
    try:
        hindi_text = clean_paragraph_pages[i]
        english_text = translate_page(hindi_text)
        translated_pages.append(english_text)
        time.sleep(0.5)   # small delay to reduce request rate
    except Exception as e:
        print(f"Error on page {i+1}: {e}")
        time.sleep(5)     # pause before stopping
        break



Translating pages:   6%|▌         | 17/280 [00:22<06:15,  1.43s/it]

Error on page 38: (429001) The server rejected the request because the client has exceeded request limits.
Code: 429001
Message: The server rejected the request because the client has exceeded request limits.


Translating pages:   6%|▌         | 17/280 [00:27<07:06,  1.62s/it]


In [63]:
# Print the last translated page index to check progress
print("Last translated page index:", len(translated_pages)-1)

Last translated page index: 36


### Slower attempt with longer delay

**What this cell did**  
- Increased the per-request delay to 1 second to reduce rate-limit hits.

**Observed behavior**  
- Still encountered 429 errors, indicating the service enforces stricter quotas than a 1s delay can avoid for this workload.

**Note**  
- This cell documents the iterative approach of increasing delays to find a safe rate.


In [64]:
# Slower attempt with 1 second delay (kept for record)
start_page = len(translated_pages)   # should be 36
end_page = 300

for i in tqdm(range(start_page, end_page), desc="Translating pages"):
    try:
        hindi_text = clean_paragraph_pages[i]
        english_text = translate_page(hindi_text)
        translated_pages.append(english_text)
        time.sleep(1)   # increased delay
    except Exception as e:
        print(f"Error on page {i+1}: {e}")
        time.sleep(5)
        break


Translating pages:   6%|▌         | 15/263 [00:23<06:03,  1.47s/it]

Error on page 53: (429001) The server rejected the request because the client has exceeded request limits.
Code: 429001
Message: The server rejected the request because the client has exceeded request limits.


Translating pages:   6%|▌         | 15/263 [00:28<07:46,  1.88s/it]


In [65]:
# Print the last translated page index after the last run
print("Last translated page index:", len(translated_pages)-1)

Last translated page index: 51


### Longer delay attempt with 3 second sleep

**What this cell did**  
- Increased the delay to 3 seconds between requests to further reduce the chance of hitting rate limits.

**Observed behavior**  
- Still produced 429 errors, but the failure point moved further into the batch. This shows the delay helps but may not be sufficient for the quota in use.

**Why we kept these experiments**  
- They document the empirical process of tuning delays and show how the service responded to different pacing strategies.


In [66]:
# Longer delay attempt with 3 second sleep (kept for record)
start_page = len(translated_pages)   # should be 51
end_page = 300

for i in tqdm(range(start_page, end_page), desc="Translating pages"):
    try:
        hindi_text = clean_paragraph_pages[i]
        english_text = translate_page(hindi_text)
        translated_pages.append(english_text)
        time.sleep(3)   # longer delay
    except Exception as e:
        print(f"Error on page {i+1}: {e}")
        time.sleep(20)
        break


Translating pages:  10%|█         | 25/248 [01:29<13:29,  3.63s/it]

Error on page 78: (429001) The server rejected the request because the client has exceeded request limits.
Code: 429001
Message: The server rejected the request because the client has exceeded request limits.


Translating pages:  10%|█         | 25/248 [01:49<16:16,  4.38s/it]


In [67]:
# Print the last translated page index after the 3s delay run
print("Last translated page index:", len(translated_pages)-1)

Last translated page index: 76


In [68]:
# Repeat of the 3 second delay attempt (kept for record)
start_page = len(translated_pages)   # should be 76
end_page = 300

for i in tqdm(range(start_page, end_page), desc="Translating pages"):
    try:
        hindi_text = clean_paragraph_pages[i]
        english_text = translate_page(hindi_text)
        translated_pages.append(english_text)
        time.sleep(3)   # keep the same delay
    except Exception as e:
        print(f"Error on page {i+1}: {e}")
        time.sleep(20)
        break


Translating pages:  11%|█         | 24/223 [01:24<11:22,  3.43s/it]

Error on page 102: (429001) The server rejected the request because the client has exceeded request limits.
Code: 429001
Message: The server rejected the request because the client has exceeded request limits.


Translating pages:  11%|█         | 24/223 [01:44<14:28,  4.36s/it]


In [69]:
# Print the last translated page index after the last run
print("Last translated page index:", len(translated_pages)-1)

Last translated page index: 100


### Translation run: paced batch

**Purpose**  
This cell continues translating pages in a paced loop. It documents the strategy used here: resume from the current `translated_pages` length, translate up to `end_page`, append results, and use a long `time.sleep` between requests to reduce the chance of rate-limit errors.

**Notes**  
- This approach is simple and worked empirically for this run, but it is not resilient to transient API errors or quota limits.  
- Keep this cell as a record of the pacing strategy used (10s delay). For production use, prefer a resumable loop with retries and backoff (added later as an alternative).


In [70]:
# Resume translating pages with a long, stable delay to reduce rate-limit hits.
# WARNING: This naive approach can still hit quotas; keep for historical record.

start_page = len(translated_pages)   # e.g., 100 at this point
end_page = 300

for i in tqdm(range(start_page, end_page), desc="Translating pages"):
    try:
        hindi_text = clean_paragraph_pages[i]
        english_text = translate_page(hindi_text)
        translated_pages.append(english_text)
        # Long delay between requests to reduce rate-limit errors
        time.sleep(10)
    except Exception as e:
        # Log the error, pause, and stop the batch to avoid repeated failures
        print(f"Error on page {i+1}: {e}")
        time.sleep(30)
        break


Translating pages: 100%|██████████| 199/199 [35:05<00:00, 10.58s/it]


In [71]:
# Print the last translated page index (zero-based index of last element)
print("Last translated page index:", len(translated_pages) - 1)


Last translated page index: 299


In [74]:
### Quick last-page test

# A small test cell to inspect the last translated page content. 
# The print is commented out to avoid accidental long outputs; uncomment to inspect a snippet.

#print(translated_pages[-1][:500])


In [75]:
# Translate the next block of pages with a stable delay.
# Note: This is another empirical run that used a 10s delay per page.

start_page = len(translated_pages)   # expected to be 300
end_page = 600

for i in tqdm(range(start_page, end_page), desc="Translating pages"):
    hindi_text = clean_paragraph_pages[i]
    english_text = translate_page(hindi_text)
    translated_pages.append(english_text)
    # Stable delay between requests
    time.sleep(10)


Translating pages: 100%|██████████| 300/300 [53:09<00:00, 10.63s/it]


In [76]:
# Print the last translated page index after the second batch
print("Last translated page index:", len(translated_pages) - 1)

Last translated page index: 599


In [77]:
# Final translation run to finish the remaining pages.
# This used the same stable delay approach (10s per page) and completed the run.

start_page = len(translated_pages)   # expected to be 600
end_page = 980   # or the actual total pages of the book

for i in tqdm(range(start_page, end_page), desc="Translating pages"):
    hindi_text = clean_paragraph_pages[i]
    english_text = translate_page(hindi_text)
    translated_pages.append(english_text)
    time.sleep(10)


Translating pages: 100%|██████████| 380/380 [1:07:21<00:00, 10.64s/it]


In [78]:
# Final progress check
# Confirm the final index after the full translation run. This should equal total_pages - 1 when the run completes successfully.

print("Last translated page index:", len(translated_pages)-1)

Last translated page index: 979


In [125]:
# Figuring out how our chapter names are placed and need to be sorted
#print(translated_pages[10])

In [97]:
#print(translated_pages[303])

In [98]:
#print(translated_pages[86])

### Chapter-title detector

**Purpose**  
Detect whether a page begins with a chapter title like "Chapter 2" or "CHAPTER 2". This helper returns the first line if it matches the pattern, otherwise `None`.

**Notes**  
- This is a conservative detector that looks only at the first line. Later we refine splitting logic to handle titles that run into the body text.


In [137]:
# Chapter-title detector: checks the first line for "Chapter <number>"
import re

def extract_chapter_title(page_text: str) -> str | None:
    
    #Return the first line if it matches a chapter title pattern, else None.
    
    lines = page_text.strip().split("\n")
    if not lines:
        return None
    first_line = lines[0].strip()

    pattern = r"^(CHAPTER|Chapter)\s+\d+"
    if re.match(pattern, first_line):
        return first_line
    return None


### Title and body splitter (simple)

**Purpose**  
Split a page into `(chapter_title, body)` when a chapter title is detected on the first line. If no chapter title is found, return `(None, page_text)`.

**Notes**  
- This helper assumes the title occupies the first line. Later cells refine splitting for titles that run into the body text.


In [138]:
# Split a page into (chapter_title, body) when the first line is a chapter title.
def split_page(page_text: str) -> tuple[str | None, str]:
    chapter_title = extract_chapter_title(page_text)

    if chapter_title:
        lines = page_text.strip().split("\n")
        body = "\n".join(lines[1:]).strip()
        return chapter_title, body

    return None, page_text


### Assemble manuscript with page markers and titles

**Purpose**  
Iterate through `translated_pages`, detect chapter titles, and build `full_text` with `=== PAGE X ===` markers and chapter titles inserted before bodies.

**Notes**  
- The first 10 pages are treated as front-matter and are not subject to chapter detection.  
- This assembly preserves page markers for debugging; later we remove them before final export.


In [139]:
# Build the full manuscript text with page markers and detected chapter titles.
full_text = ""

for i, page in enumerate(translated_pages, start=1):

    # Skip chapter detection for the first 10 pages (index/front-matter)
    if i <= 10:
        full_text += f"=== PAGE {i} ===\n"
        full_text += page + "\n\n"
        continue

    # For pages 11 onward, detect chapter titles and split
    chapter_title, body = split_page(page)

    full_text += f"=== PAGE {i} ===\n"

    if chapter_title:
        full_text += chapter_title + "\n\n"

    full_text += body + "\n\n"


In [142]:
# Testing the Title separation
# Title splitting refinement: single-page test

# The simple first-line detector failed for titles that run into the body. This cell introduces a refined `split_page` that:
# matches a chapter start pattern,
# searches for a likely split point (two spaces or a period followed by a capital letter),
# returns `(chapter_title, body)` accordingly.

# A single-page test follows to verify the split behavior.


# print(full_text[40000:50000])

# Failed, so we shall try on one single page first

In [152]:
def split_page(page_text: str) -> tuple[str | None, str]:
    
    # Refined splitter:
    # If the page starts with 'Chapter <num>' it tries to find a split point
    # where the title ends and the body begins.
    # split heuristics: two spaces '  ' OR a period followed by a space and a capital letter '. [A-Z]'.
    
    text = page_text.strip()

    # Match the chapter number at the start
    chapter_start = re.match(r"^(Chapter|CHAPTER)\s+\d+[:.]?", text)
    if not chapter_start:
        return None, page_text

    # Find where the title likely ends
    split_match = re.search(r"  |\. [A-Z]", text)

    if split_match:
        end = split_match.start()
        chapter_title = text[:end].strip()
        body = text[end:].lstrip()
        return chapter_title, body

    # If no split point found, treat the whole text as title and return empty body
    return text, ""

# Single-page test to verify the refined splitter
chapter_title, body = split_page(translated_pages[23])
print("TITLE:", chapter_title)
print("BODY START:", body[:200])


TITLE: Chapter 2: The Soul and Vidyapraveda of the Vedas There is a great discussion of knowledge and avidya in the Upanishads
BODY START: . Nowadays, the meanings of these two words are no longer of the Upanishadic period but have been adapted into the accepted form in the medieval Sankhya philosophy, which makes it difficult to underst


### Extract raw chapter numbers from the front-matter

**Purpose**  
Scan the first N index pages (front-matter) to collect any lines that look like chapter headings such as `Chapter 3`, `Chapter 4`, etc. This produces a raw list that may contain OCR artifacts (e.g., `47.` or `1.5`) which we will normalize in the next step.

**Inputs**  
- `translated_pages` (list of translated page strings)

**Outputs**  
- `raw_chapter_numbers` (list of strings as found in the index)


In [173]:
# Extract candidate chapter numbers from the first 10 pages (index/front-matter).
# This is intentionally permissive to capture OCR artifacts like "47." or "1.5".

raw_chapter_numbers = []

for i in range(10):  # first 10 pages = index/front-matter
    page = translated_pages[i]
    lines = page.split("\n")
    for line in lines:
        line = line.strip()
        # Match "Chapter <number>" where number may include dots (OCR noise)
        match = re.match(r"^(Chapter|CHAPTER)\s+([\d\.]+)", line)
        if match:
            raw_chapter_numbers.append(match.group(2))

print(raw_chapter_numbers)


['3', '47.', '6', '7', '8', '13', '14', '1.5', '16', '16', '20', '21', '27']


In [174]:
# Map known OCR mistakes to the intended chapter numbers and normalize the list.
manual_number_fixes = {
    "47.": "4",   # OCR misread "4" as "47."
    "1.5": "15",  # OCR misread "15" as "1.5"
}

def normalize_chapter_number(num: str) -> str:
    num = num.strip()
    if num in manual_number_fixes:
        return manual_number_fixes[num]
    return num

cleaned_numbers = [normalize_chapter_number(n) for n in raw_chapter_numbers]
print(cleaned_numbers)


['3', '4', '6', '7', '8', '13', '14', '15', '16', '16', '20', '21', '27']


### Ensure expected chapter numbers and remove duplicates

**Purpose**  
- Insert any missing chapter numbers that you know should be present (e.g., chapter 2).  
- Fix individual entries by index when OCR produced numeric errors.  
- Remove duplicates while preserving order.


In [175]:
# Insert known missing chapters and correct specific indices if needed.

# If chapter "2" is expected but not found in the index, insert it at the front
if "2" not in cleaned_numbers:
    cleaned_numbers.insert(0, "2")
print(cleaned_numbers)

# Example manual correction by position (used when a specific index was misread)
# Replace the 12th element (index 11) with 17 if we know it should be 17
cleaned_numbers[11] = 17
print(cleaned_numbers)

# Remove duplicates while preserving order
cleaned_numbers = list(dict.fromkeys(cleaned_numbers))
print(cleaned_numbers)


['2', '3', '4', '6', '7', '8', '13', '14', '15', '16', '16', '20', '21', '27']


['2', '3', '4', '6', '7', '8', '13', '14', '15', '16', '16', 17, '21', '27']


['2', '3', '4', '6', '7', '8', '13', '14', '15', '16', 17, '21', '27']


In [180]:
# Human-verified title overrides for chapters found in the index.
# Add or edit entries as you confirm correct chapter titles.

KNOWN_TITLES = {
    "2":  "Chapter 2: The Soul and Vidyapraveda of the Vedas",
    "3":  "Chapter 3: The Seven Qualifications of Vedic Interpretation",
    "13": "Chapter 13: List of Vedic Secrets on Brahman",
    "14": "Chapter 14: The Ana-Vidya of the Vedas",
    "16": "Chapter 16: Fire in the Vedas",
    "20": "Chapter 20: Ghrit and Ajya",
    "27": "Chapter 27: The Sages"
}



### Detect chapter pages and perform a simple split

**Purpose**  
- `detect_chapter_by_number` checks whether a page begins with a known chapter number (from `cleaned_numbers`).  
- `split_page_simple` extracts the title as the first matching line and returns the remainder as the body.  
- `extract_title_with_overrides` uses `KNOWN_TITLES` when available to ensure clean titles.

**Notes**  
- This is a conservative approach: it looks for `Chapter <num>` at the start of the page.
- Later cells refine splitting for titles that run into the body text.


In [184]:
# Detect chapter pages by number and split title/body simply when the title is on the first line.

def detect_chapter_by_number(page_text: str, chapter_numbers: list) -> str | None:
    """
    Return the chapter number (as string) if the page starts with 'Chapter <num>' where <num> is in chapter_numbers.
    """
    text = page_text.strip()
    for num in chapter_numbers:
        pattern = rf"^(Chapter|CHAPTER)\s+{re.escape(str(num))}\b"
        if re.match(pattern, text):
            return str(num)
    return None

def split_page_simple(page_text: str, chapter_num: str) -> tuple[str | None, str]:
    """
    If the page starts with 'Chapter <chapter_num>', return (title, body).
    The title is taken as the matched prefix; the body is the rest of the text.
    """
    text = page_text.strip()
    pattern = rf"^(Chapter|CHAPTER)\s+{re.escape(chapter_num)}.*"
    match = re.match(pattern, text)

    if match:
        title = match.group().strip()
        body = text[len(title):].lstrip()
        return title, body

    return None, page_text

def extract_title_with_overrides(page_text: str, chapter_num: str) -> tuple[str, str]:
    """
    If a clean title exists in KNOWN_TITLES, use it and remove the 'Chapter <num>' prefix from the page to form the body.
    Otherwise, fall back to split_page_simple.
    """
    text = page_text.strip()

    if chapter_num in KNOWN_TITLES:
        title = KNOWN_TITLES[chapter_num]

        # Remove the 'Chapter <num>' prefix if present, then strip common punctuation
        prefix_pattern = rf"^(Chapter|CHAPTER)\s+{re.escape(chapter_num)}\s*"
        prefix_match = re.match(prefix_pattern, text)
        if prefix_match:
            body = text[prefix_match.end():].lstrip(" :.-").lstrip()
        else:
            body = text

        return title, body

    return split_page_simple(page_text, chapter_num)



In [185]:
# Test the detection and override logic on a sample page (PAGE 24 in this example)

test_page_index = 23  # PAGE 24 (0-based index)
page = translated_pages[test_page_index]

chapter_num = detect_chapter_by_number(page, cleaned_numbers)
print("Detected chapter number:", chapter_num)

if chapter_num:
    title, body = extract_title_with_overrides(page, chapter_num)
    print("\n=== TITLE ===")
    print(title)
    print("\n=== BODY START ===")
    print(body[:500])  # preview first 500 characters of the body
else:
    print("No chapter detected on this page.")


Detected chapter number: 2

=== TITLE ===
Chapter 2: The Soul and Vidyapraveda of the Vedas

=== BODY START ===
The Soul and Vidyapraveda of the Vedas There is a great discussion of knowledge and avidya in the Upanishads. Nowadays, the meanings of these two words are no longer of the Upanishadic period but have been adapted into the accepted form in the medieval Sankhya philosophy, which makes it difficult to understand the symbolic Vedic meaning. In the Vedic age the words Vidya and Abiyad were used to refer to the elements of the Purva and Uttaradha of the Vedic Darshana. The spiritual creation of the f


### Assemble the full manuscript with detected titles

**Purpose**  
Iterate through all translated pages, detect chapter pages using `cleaned_numbers`, apply title overrides when available, and build `full_text` with `=== PAGE X ===` markers for debugging.

**Notes**  
- The first 10 pages are treated as front-matter and are not subject to chapter detection.  
- We keep page markers during development for traceability; they can be removed before final export.


In [186]:
# Build the full manuscript text with page markers and detected chapter titles.

full_text = ""

for i, page in enumerate(translated_pages, start=1):

    # Skip index/front-matter pages (first 10)
    if i <= 10:
        full_text += f"=== PAGE {i} ===\n{page}\n\n"
        continue

    chapter_num = detect_chapter_by_number(page, cleaned_numbers)

    if chapter_num:
        title, body = extract_title_with_overrides(page, chapter_num)
        full_text += f"=== PAGE {i} ===\n{title}\n\n{body}\n\n"
    else:
        full_text += f"=== PAGE {i} ===\n{page}\n\n"


In [188]:
# Testing the Title separation
#print(full_text[40000:50000])

### Gentle English cleaning pass

**Purpose**  
A non-destructive cleaning function to fix common translation/OCR artifacts:
- remove repeated words (e.g., "no exaggeration, no exaggeration"),
- collapse multiple spaces,
- remove stray spaces before punctuation,
- capitalize the first letter after newlines.

**Notes**  
This function is intentionally conservative to avoid changing meaning. Use it on titles and bodies before final assembly.


In [192]:
# Define a gentle English cleaning function using regular expressions.

def clean_english(text: str) -> str:
    cleaned = text

    # Remove repeated words like "no exaggeration, no exaggeration"
    cleaned = re.sub(r'\b(\w+)( \1\b)+', r'\1', cleaned)

    # Collapse multiple spaces into one
    cleaned = re.sub(r' {2,}', ' ', cleaned)

    # Remove space before punctuation (e.g., "word , next" -> "word, next")
    cleaned = re.sub(r'\s+([,.!?;:])', r'\1', cleaned)

    # Capitalize first letter after newlines and at the start of the text
    cleaned = re.sub(r'(^|\n)([a-z])', lambda m: m.group(1) + m.group(2).upper(), cleaned)

    return cleaned.strip()



### Insert chapter break markers and prepare final text

**Purpose**  
- Walk through the assembled `full_text` (or page blocks) and insert `=== CHAPTER BREAK ===` markers before pages that start with a chapter title.  
- Clean English on each page block.  
- Finally, remove the `=== PAGE X ===` debug markers so the manuscript flows naturally.

**Notes**  
- We keep `=== CHAPTER BREAK ===` markers so they can be replaced with real page breaks in Word/Google Docs (Find & Replace → Manual Page Break / ^m).


In [200]:
# Build a list of final page blocks with chapter-break markers and then join them into final_text.

final_pages = []

# `pages` here refers to the page-blocks created earlier (e.g., from full_text.split("=== PAGE "))
# If you have `full_text`, split it into page blocks first:
# pages = full_text.split("=== PAGE ")

for p in pages:
    if not p.strip():
        continue

    # Each page block begins with "<page_number> ===" because we split on "=== PAGE "
    page_number, content = p.split(" ===", 1)
    cleaned_content = clean_english(content)

    # If the cleaned content starts with a chapter title, insert a chapter break marker
    starts_with_chapter = cleaned_content.strip().startswith("Chapter ")

    if starts_with_chapter:
        final_pages.append("=== CHAPTER BREAK ===\n")

    final_pages.append(f"=== PAGE {page_number} ==={cleaned_content}\n")

final_text = "\n".join(final_pages)

# Remove all page markers like "=== PAGE 1 ===" before final export
final_text_cleaned = re.sub(r'^=== PAGE \d+ ===\s*', '', final_text, flags=re.MULTILINE)



In [202]:
# Save the final cleaned manuscript to a UTF-8 text file
# The file contains chapter break markers === CHAPTER BREAK ===
# which you will replace with real page breaks in Word/Google Docs.

with open("final_book.txt", "w", encoding="utf-8") as f:
    f.write(final_text_cleaned)


### Save final manuscript to file

**What this cell does**  
Writes `final_text_cleaned` to `final_book.txt` using UTF-8 encoding. The file contains the cleaned body text and `=== CHAPTER BREAK ===` markers where each chapter should start on a new page.

**Why UTF-8**  
Use UTF-8 to preserve special characters and avoid corrupted text when opening in Word or Google Docs.

**Next step**  
Open `final_book.txt` in Word or Google Docs and follow the manual formatting steps below to produce the final PDF.


### Final notes

The manuscript has been saved as `final_book.txt`. Open it in Word or Google Docs with UTF-8 encoding, replace `=== CHAPTER BREAK ===` with real page breaks, apply Heading styles, and export to PDF. Keep this notebook as the canonical pipeline: it documents OCR extraction, paragraph cleaning, translation, chapter detection, and final assembly. For reproducible runs or large books, consider using the resumable translation loop (added later) to avoid API rate limits.


### Robust Resumable Translation

**Purpose**  
Provide a safe, resumable, and rate‑aware translation runner that replaces long ad‑hoc loops. This cell processes `clean_paragraph_pages` in small batches, saves progress after each successful page, and retries transient failures with exponential backoff and jitter.

**What it replaces**  
- The manual loops that used fixed `time.sleep` values (0.5s, 1s, 3s, 10s) and required manual restarts.  
- Replaces repeated edits to `start_page` and manual re‑runs after 429 errors.

**How it works**  
1. Loads any previously saved progress from `translation_progress.json` and `translated_pages_partial.json`.  
2. Processes a configurable batch of pages (`BATCH_SIZE`) and saves progress after each page.  
3. On transient errors, retries up to `MAX_RETRIES` with exponential backoff and small random jitter.  
4. Writes progress atomically to avoid corrupt files and allows safe interruption and resume.

**Files created**  
- `translation_progress.json` — last completed page index.  
- `translated_pages_partial.json` — partial list of translated pages (UTF‑8).

**Quick usage notes**  
- Tune `BATCH_SIZE` and `DELAY_SECONDS` before running. Start small (e.g., `BATCH_SIZE=10`, `DELAY_SECONDS=10.0`) to validate behavior.  
- Re‑run the same cell to continue from the saved index.  
- Keep the original experimental cells and error traces for reproducibility.

**Status**  
This cell is provided as a recommended alternative and has not been executed in this notebook. Run it once with conservative settings to confirm it behaves as expected in your environment.


In [ ]:
# Robust resumable translation loop (runnable)
# NOTE: This cell assumes `clean_paragraph_pages` and `translate_page(text)` are already defined.
# NOTE: This cell has not been run in my environment. Tune parameters before running.
# Start conservatively for your first run: BATCH_SIZE=10, DELAY_SECONDS=2.0, MAX_RETRIES=4.

import json
import os
import time
import random
from tqdm import tqdm
from typing import List

# === Configurable parameters (tune these conservatively for first run) ===
START_INDEX_FILE = "translation_progress.json"       # stores last completed index
OUTPUT_TRANSLATED_FILE = "translated_pages_partial.json"
BATCH_SIZE = 10             # pages to process per run (start small)
DELAY_SECONDS = 2.0         # base delay between successful requests
MAX_RETRIES = 4             # retries per page on transient errors
BACKOFF_FACTOR = 2.0        # exponential backoff multiplier
JITTER_FACTOR = 0.1         # fraction of backoff used as jitter

# === Load or initialize translated_pages ===
if os.path.exists(OUTPUT_TRANSLATED_FILE):
    with open(OUTPUT_TRANSLATED_FILE, "r", encoding="utf-8") as f:
        translated_pages: List[str] = json.load(f)
else:
    translated_pages = []

# === Determine resume index ===
start_index = 0
if os.path.exists(START_INDEX_FILE):
    with open(START_INDEX_FILE, "r", encoding="utf-8") as f:
        meta = json.load(f)
        start_index = meta.get("last_index", len(translated_pages))

total_pages = len(clean_paragraph_pages)
end_index = min(total_pages, start_index + BATCH_SIZE)

print(f"Resuming translation from index {start_index} to {end_index - 1} (total pages: {total_pages})")

# === Helpers ===
def atomic_write(path: str, data: str, mode: str = "w", encoding: str = "utf-8") -> None:
    """Write data to a temp file and atomically replace the target file."""
    tmp = path + ".tmp"
    with open(tmp, mode, encoding=encoding) as f:
        f.write(data)
    os.replace(tmp, path)

def save_progress(last_index: int, translated_list: List[str]) -> None:
    """Save last completed index and translated pages atomically."""
    atomic_write(START_INDEX_FILE, json.dumps({"last_index": last_index}))
    atomic_write(OUTPUT_TRANSLATED_FILE, json.dumps(translated_list, ensure_ascii=False))

def compute_backoff(attempt: int) -> float:
    """Compute exponential backoff with small jitter (seconds)."""
    backoff = (BACKOFF_FACTOR ** (attempt - 1)) * DELAY_SECONDS
    jitter = backoff * JITTER_FACTOR * (random.random() * 2 - 1)  # +/- jitter
    return max(1.0, backoff + jitter)

# === Main loop with retries and backoff ===
try:
    for i in tqdm(range(start_index, end_index), desc="Translating pages"):
        hindi_text = clean_paragraph_pages[i]
        attempt = 0
        while attempt <= MAX_RETRIES:
            try:
                # Call the translation function defined earlier in the notebook
                english_text = translate_page(hindi_text)
                translated_pages.append(english_text)

                # Save progress after each successful page
                save_progress(i + 1, translated_pages)

                # Respect base delay to avoid hitting rate limits
                time.sleep(DELAY_SECONDS)
                break  # success -> exit retry loop

            except Exception as e:
                attempt += 1
                print(f"Error on page {i+1} attempt {attempt}: {e}")

                if attempt > MAX_RETRIES:
                    # Save progress and stop the batch so you can inspect the error
                    print(f"Max retries exceeded for page {i+1}. Saving progress and stopping batch.")
                    save_progress(i, translated_pages)
                    raise

                # Exponential backoff with jitter before retrying
                sleep_time = compute_backoff(attempt)
                print(f"Sleeping for {sleep_time:.1f}s before retrying page {i+1}")
                time.sleep(sleep_time)

    print("Batch translation complete. Run this cell again to continue the next batch.")

except Exception as final_e:
    # Ensure progress is saved on unexpected exceptions
    print("Batch stopped due to exception:", final_e)
    save_progress(i, translated_pages)
    raise
